<a href="https://colab.research.google.com/github/faisu6339-glitch/LLMs/blob/main/Seq2Seq_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Seq2Seq LSTM Explained

Seq2Seq (Sequence-to-Sequence) LSTM is a powerful neural network architecture commonly used for tasks that involve mapping an input sequence to an output sequence, where both sequences can have different lengths. A classic example is machine translation, where an input sentence in one language is translated into an output sentence in another language.

Here's a breakdown of its components and how it works:

**1. Encoder-Decoder Architecture:**

*   **Encoder:** The encoder processes the input sequence (e.g., source language sentence) word by word (or token by token). It reads the entire input sequence and compresses all the information into a fixed-size vector, often called a "context vector" or "thought vector." This context vector is intended to capture the semantic meaning of the entire input sequence.
    *   **LSTM Cells in Encoder:** The encoder typically uses Recurrent Neural Network (RNN) layers, and LSTMs (Long Short-Term Memory) are particularly effective here. Each LSTM cell takes the current input token and the previous hidden state (and cell state) to produce a new hidden state and cell state. The final hidden state (or a combination of hidden and cell states) of the encoder is usually taken as the context vector.

*   **Decoder:** The decoder then takes this context vector as its initial hidden state and generates the output sequence (e.g., target language sentence) one word at a time. During training, it's often fed the previously generated correct output token as input to predict the next token (teacher forcing).
    *   **LSTM Cells in Decoder:** Similar to the encoder, the decoder also uses LSTM cells. At each time step, it takes the previous hidden state, the previous output token, and the context vector to produce the next hidden state and predict the next output token.

**2. Key Characteristics of LSTMs:**

*   **Memory Cells:** LSTMs have a "cell state" that acts as a conveyor belt, carrying information across many time steps. This allows them to remember information for long periods.
*   **Gates:** LSTMs use three types of gates to control the flow of information into and out of the memory cell:
    *   **Forget Gate:** Decides what information to throw away from the cell state.
    *   **Input Gate:** Decides what new information to store in the cell state.
    *   **Output Gate:** Decides what part of the cell state to output as the hidden state.

**3. Working Process (Example: Machine Translation):**

1.  **Input:** An English sentence, e.g., "I am a student."
2.  **Encoder:** The encoder LSTM processes "I", then "am", then "a", then "student", updating its hidden and cell states at each step. After processing "student", it produces a final hidden state (the context vector) that summarizes the meaning of the entire English sentence.
3.  **Decoder:** The decoder LSTM receives this context vector. It then starts generating the translated sentence:
    *   It might receive a special `<s>` (start-of-sequence) token as its first input.
    *   Using the context vector and `<s>`, it predicts the first word of the French translation, e.g., "Je".
    *   Then, using the context vector and "Je" (or the correct output "Je" during training), it predicts "suis".
    *   This continues until it predicts an `</s>` (end-of-sequence) token, signaling the end of the translation.

**4. Limitations and Enhancements (Attention Mechanism):**

*   **Fixed-size Context Vector:** A major limitation of early Seq2Seq models was that the encoder had to compress all information into a single fixed-size context vector, regardless of the input sequence length. This could lead to information loss, especially for very long sequences.
*   **Attention Mechanism:** To address this, the **attention mechanism** was introduced. Instead of forcing the encoder to compress everything into one vector, attention allows the decoder to "look back" at the relevant parts of the input sequence at each step of generating the output. This significantly improved the performance of Seq2Seq models, particularly for longer sequences, and is now a standard component in advanced sequence models like Transformers.

In summary, Seq2Seq LSTM provides a robust framework for sequence transduction tasks by using separate LSTMs for encoding and decoding, leveraging the memory capabilities of LSTMs to handle variable-length sequences and capture complex dependencies. With the addition of attention, it forms the basis for many state-of-the-art models in natural language processing.

# Program 1: Reverse Sequence using Seq2Seq LSTM

Input  : 1 2 3
Output : 3 2 1

Input  : 4 5 6
Output : 6 5 4

In [32]:
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.utils import to_categorical

In [33]:
X = np.array([
    [1,2,3],
    [2,3,4],
    [3,4,5],
    [4,5,6],
    [5,6,7],
    [6,7,8],
    [7,8,9],
    [8,9,1],
    [9,1,2]
])

Y = np.array([
    [3,2,1],
    [4,3,2],
    [5,4,3],
    [6,5,4],
    [7,6,5],
    [8,7,6],
    [9,8,7],
    [1,9,8],
    [2,1,9]
])

In [34]:
vocab_size = 10
X = to_categorical(X, num_classes=vocab_size)
Y = to_categorical(Y, num_classes=vocab_size)

# Build Encoder

In [35]:
latent_dim = 64

encoder_inputs = Input(shape=(3, vocab_size))

encoder_lstm = LSTM(
    latent_dim,
    return_state=True
)

_, state_h, state_c = encoder_lstm(encoder_inputs)

encoder_states = [state_h, state_c]

# : Build Decoder

In [36]:
decoder_inputs = Input(shape=(3, vocab_size))

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_inputs,
    initial_state=encoder_states
)

decoder_dense = Dense(
    vocab_size,
    activation="softmax"
)

decoder_outputs = decoder_dense(decoder_outputs)

In [37]:

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
decoder_input = np.zeros_like(Y)

decoder_input[:,1:,:] = Y[:,:-1,:]

In [38]:
model.fit(
    [X, decoder_input],
    Y,
    epochs=300,
    batch_size=2,
    verbose=1
)

Epoch 1/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.0741 - loss: 2.3103
Epoch 2/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2593 - loss: 2.2911    
Epoch 3/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3704 - loss: 2.2732
Epoch 4/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5926 - loss: 2.2569
Epoch 5/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6296 - loss: 2.2386
Epoch 6/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6667 - loss: 2.2201
Epoch 7/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6667 - loss: 2.1999
Epoch 8/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6667 - loss: 2.1778
Epoch 9/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6667 - loss: 2.1540
Epoch 10/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6667 - loss: 2.1223
Epoch 11/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6667 - loss: 2.0837
Epoch 12/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6667 

In [39]:
prediction = model.predict(
    [X, decoder_input]
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 413ms/step


# Convert Prediction

In [40]:
predicted = np.argmax(prediction, axis=-1)

actual = np.argmax(Y, axis=-1)

original = np.argmax(X, axis=-1)

# Display Results

In [41]:
for i in range(len(original)):
    print("Input     :", original[i])
    print("Expected  :", actual[i])
    print("Predicted :", predicted[i])
    print("-"*40)

Input     : [1 2 3]
Expected  : [3 2 1]
Predicted : [3 2 1]
----------------------------------------
Input     : [2 3 4]
Expected  : [4 3 2]
Predicted : [4 3 2]
----------------------------------------
Input     : [3 4 5]
Expected  : [5 4 3]
Predicted : [5 4 3]
----------------------------------------
Input     : [4 5 6]
Expected  : [6 5 4]
Predicted : [6 5 4]
----------------------------------------
Input     : [5 6 7]
Expected  : [7 6 5]
Predicted : [7 6 5]
----------------------------------------
Input     : [6 7 8]
Expected  : [8 7 6]
Predicted : [8 7 6]
----------------------------------------
Input     : [7 8 9]
Expected  : [9 8 7]
Predicted : [9 8 7]
----------------------------------------
Input     : [8 9 1]
Expected  : [1 9 8]
Predicted : [1 9 8]
----------------------------------------
Input     : [9 1 2]
Expected  : [2 1 9]
Predicted : [2 1 9]
----------------------------------------


# Program 2. English → French Translation (LSTM Encoder-Decoder)

In [1]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
english_sentences = [
    "hello",
    "how are you",
    "good morning",
    "good night",
    "thank you",
    "i love you",
    "what is your name",
    "where are you",
    "see you later",
    "i am happy"
]

french_sentences = [
    "bonjour",
    "comment allez vous",
    "bonjour",
    "bonne nuit",
    "merci",
    "je t aime",
    "comment tu t appelles",
    "ou etes vous",
    "a plus tard",
    "je suis heureux"
]

# Add Start and End Tokens

In [3]:
decoder_input_text = ["<start> " + txt for txt in french_sentences]
decoder_output_text = [txt + " <end>" for txt in french_sentences]

# Tokenization

**English Tokenizer**

In [4]:
eng_tokenizer = Tokenizer(filters='')
eng_tokenizer.fit_on_texts(english_sentences)

encoder_sequences = eng_tokenizer.texts_to_sequences(english_sentences)

encoder_vocab_size = len(eng_tokenizer.word_index) + 1

**French Tokenizer**

In [5]:
fra_tokenizer = Tokenizer(filters='')
fra_tokenizer.fit_on_texts(decoder_input_text + decoder_output_text)

decoder_input_sequences = fra_tokenizer.texts_to_sequences(decoder_input_text)
decoder_output_sequences = fra_tokenizer.texts_to_sequences(decoder_output_text)

decoder_vocab_size = len(fra_tokenizer.word_index) + 1

In [24]:
print(fra_tokenizer.word_index)

{'<start>': 1, '<end>': 2, 'bonjour': 3, 'comment': 4, 'vous': 5, 'je': 6, 't': 7, 'allez': 8, 'bonne': 9, 'nuit': 10, 'merci': 11, 'aime': 12, 'tu': 13, 'appelles': 14, 'ou': 15, 'etes': 16, 'a': 17, 'plus': 18, 'tard': 19, 'suis': 20, 'heureux': 21}


# Padding

In [6]:
max_encoder_len = max(len(seq) for seq in encoder_sequences)
max_decoder_len = max(len(seq) for seq in decoder_input_sequences)

encoder_input_data = pad_sequences(
    encoder_sequences,
    maxlen=max_encoder_len,
    padding='post'
)

decoder_input_data = pad_sequences(
    decoder_input_sequences,
    maxlen=max_decoder_len,
    padding='post'
)

decoder_output_data = pad_sequences(
    decoder_output_sequences,
    maxlen=max_decoder_len,
    padding='post'
)

In [7]:
decoder_output_data = np.expand_dims(decoder_output_data, -1)

# Hyperparameters

In [8]:
embedding_dim = 128
latent_dim = 256

# Encoder

In [9]:
encoder_inputs = Input(shape=(None,))

encoder_embedding = Embedding(
    input_dim=encoder_vocab_size,
    output_dim=embedding_dim
)(encoder_inputs)

encoder_lstm = LSTM(
    latent_dim,
    return_state=True
)

_, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]

# Decoder

In [10]:
decoder_inputs = Input(shape=(None,))

decoder_embedding = Embedding(
    input_dim=decoder_vocab_size,
    output_dim=embedding_dim
)(decoder_inputs)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

# Output Layer

In [11]:
decoder_dense = Dense(
    decoder_vocab_size,
    activation="softmax"
)

decoder_outputs = decoder_dense(decoder_outputs)

# Build Model

In [12]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

In [13]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [14]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 128) │      2,560 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 128) │      2,816 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    394,240 │ embedding[0][0]   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    394,240 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 22)  │      5,654 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 799,510 (3.05 MB)

 Trainable params: 799,510 (3.05 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_output_data,
    batch_size=2,
    epochs=300,
    verbose=1
)

Epoch 1/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - accuracy: 0.2400 - loss: 3.0618
Epoch 2/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.3200 - loss: 2.8414
Epoch 3/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.3200 - loss: 2.2985
Epoch 4/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.3200 - loss: 2.0968
Epoch 5/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.4800 - loss: 1.9223
Epoch 6/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.4800 - loss: 1.8545
Epoch 7/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.4000 - loss: 1.7882
Epoch 8/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.4600 - loss: 1.7063
Epoch 9/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.5200 - loss: 1.6267
Epoch 10/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.5200 - loss: 1.5646
Epoch 11/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.5400 - loss: 1.5051
Epoch 12/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6000 - lo

# Encoder Inference Model

In [16]:
encoder_model = Model(
    encoder_inputs,
    encoder_states
)

# Decoder Inference Model

In [18]:
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

decoder_emb2 = decoder_embedding

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    decoder_emb2,
    initial_state=decoder_states_inputs
)

decoder_states = [state_h2, state_c2]

decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states
)

# Reverse Dictionaries

In [19]:
reverse_fra_index = {
    i: word
    for word, i in fra_tokenizer.word_index.items()
}

reverse_eng_index = {
    i: word
    for word, i in eng_tokenizer.word_index.items()
}

# Translation Function

In [25]:
def translate(sentence):

    seq = eng_tokenizer.texts_to_sequences([sentence])

    seq = pad_sequences(
        seq,
        maxlen=max_encoder_len,
        padding='post'
    )

    states = encoder_model.predict(seq, verbose=0)

    target_seq = np.zeros((1, 1))

    target_seq[0, 0] = fra_tokenizer.word_index["<start>"]

    stop = False

    decoded_sentence = ""

    while not stop:

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states,
            verbose=0
        )

        sampled_token = np.argmax(output_tokens[0, -1, :])

        sampled_word = reverse_fra_index.get(sampled_token, "")

        if sampled_word == "<end>" or len(decoded_sentence.split()) > 20:
            stop = True
        else:
            decoded_sentence += sampled_word + " "

        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token

        states = [h, c]

    return decoded_sentence

In [26]:
print(translate("hello"))
print(translate("thank you"))
print(translate("good night"))
print(translate("i love you"))

bonjour 
merci 
bonne nuit 
je t aime 


These translations match the training dataset, so your Encoder–Decoder LSTM has successfully learned the mappings.

### What you've just built

YouYou have implemented the classical Sequence-to-Sequence (Seq2Seq) architecture.

```
English Sentence
        │
        ▼
+-------------------+
|  Encoder          |
|  Embedding        |
+-------------------+
        │
        ▼
+-------------------+
|      LSTM         |
+-------------------+
        │
        ▼
Hidden State (h,c)
        │
        ▼
+-------------------+
|  Decoder          |
|  Embedding        |
+-------------------+
        │
        ▼
+-------------------+
|      LSTM         |
+-------------------+
        │
        ▼
+-------------------+
|      Dense        |
+-------------------+
        │
        ▼
French Translation
```

### How the model works

**Step 1: Encoder**

Input:

`hello`

↓

`Embedding`

↓

`[0.45, -0.81, 0.23, ...]`

↓

`LSTM`

↓

Produces two vectors:

`Hidden State (h)`
`Cell State (c)`

These vectors summarize the meaning of the input sentence.

**Step 2: Decoder**

The decoder starts with the special token:

`<start>`

Using the encoder's hidden and cell states, it predicts the next word:

`bonjour`

Then the predicted word is fed back into the decoder:

`<start>`
      ↓
`bonjour`
      ↓
`<end>`

**During Training (Teacher Forcing)**

The decoder is given the correct previous word.

Input

`<start>`

↓

`bonjour`

↓

`<end>`

Instead of using its own predictions, it learns from the correct sequence, which speeds up convergence.

**During Inference (Prediction)**

The decoder uses its own previous prediction:

`<start>`

↓

`bonjour`

↓

`<end>`

This continues until the `<end>` token is produced.

### Current Limitation

Your dataset contains only 10 sentence pairs, so the model has essentially memorized them.

Try a sentence that wasn't in training:

```python
print(translate("good morning"))
```

Expected:

`bonjour`

Try:

```python
print(translate("where are you"))
```

Expected:

`ou etes vous`

But if you try:

```python
print(translate("good evening"))
```

the model may produce an incorrect translation or repeat a known sentence because it has never seen that input.